# 17 — N-Grams

N-grams are contiguous sequences of n tokens — "machine", "machine learning", "machine learning is" for n=1,2,3. They restore a slice of word order that the bag of words (Ch. 15) threw away, at the cost of a larger, sparser vocabulary.

**Why it matters for resumes / ATS:** the most important resume terms are multi-word: "machine learning", "data science", "computer vision". A unigram-only index can't distinguish "deep learning" from "deep" + "learning" — n-grams capture the exact phrases recruiters and ATS dictionaries look for.

**Goal:** Capture word order and multi-word expressions using n-grams.

This chapter covers the three n-gram flavors (unigrams/bigrams/trigrams), skill detection with `ngram_range=(1,3)`, and a character-level overlap score that stays robust to word-level differences. The takeaway: `(1,2)` is the sweet spot for most resume work.

## 1. Unigrams, Bigrams, Trigrams

`CountVectorizer(ngram_range=(n, n))` builds a vocabulary of n-word windows. Higher n captures more context but fragments fast — a 10-word sentence yields 10 unigrams but only 8 trigrams, and the vocabulary explodes with corpus size.

**What the code does:** fits three vectorizers on one sentence and prints the first 10 features of each n.
- 1-grams (7): `['and', 'deep', 'fun', 'is', 'learning', 'machine', 'powerful']` — unique tokens.
- 2-grams (7): `['and deep', 'deep learning', 'fun and', 'is fun', ...]` — note `machine learning` survives as a unit.
- 3-grams (7): `['and deep learning', 'deep learning is', ...]` — longer context, zero hits outside the sentence.

**Try it:** "machine learning" appears as a bigram while its unigrams (`machine`, `learning`) also exist separately — that redundancy is why n-gram vocabularies grow so fast.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer
doc = "machine learning is fun and deep learning is powerful"
for n in [1, 2, 3]:
    v = CountVectorizer(ngram_range=(n, n)).fit([doc])
    print(f"\n{n}-grams ({len(v.get_feature_names_out())}):")
    print(f"  {v.get_feature_names_out()[:10]}")


1-grams (7):
  ['and' 'deep' 'fun' 'is' 'learning' 'machine' 'powerful']

2-grams (7):
  ['and deep' 'deep learning' 'fun and' 'is fun' 'is powerful' 'learning is'
 'machine learning']

3-grams (7):
  ['and deep learning' 'deep learning is' 'fun and deep' 'is fun and'
 'learning is fun' 'learning is powerful' 'machine learning is']


## 2. N-Grams for Skill Detection

Scoring n-gram features by corpus frequency surfaces the multi-word skills: "machine learning" and "deep learning" should rank high, while accidental bigrams ("experience natural") should sink. This is the feature-extraction step that feeds classifiers in later chapters.

**What the code does:** fits `CountVectorizer(ngram_range=(1, 3), stop_words="english")` on three skill sentences and ranks features by summed counts.
- "learning" leads with count 2, followed by single-hit bigrams and trigrams like `computer vision`, `data science`, `deep learning`.

**Try it (known issue):** this cell calls `np.array(...)` but never imports numpy in this notebook, so a fresh kernel raises `NameError: name 'np' is not defined`. Add `import numpy as np` at the top to see the ranking — the intent is a summed-count ranking of n-gram features.

In [3]:
import numpy as np 
docs = [
    "Expert in machine learning and deep learning",
    "Experience with natural language processing",
    "Skilled in data science and computer vision",
]
vec = CountVectorizer(ngram_range=(1, 3), stop_words="english")
X = vec.fit_transform(docs)
features = vec.get_feature_names_out()
scores = np.array(X.sum(axis=0)).flatten()
print("Top n-gram features:")
for f, s in sorted(zip(features, scores), key=lambda x: -x[1])[:10]:
    print(f"  {f:25s} {s}")

Top n-gram features:
  learning                  2
  computer                  1
  computer vision           1
  data                      1
  data science              1
  data science computer     1
  deep                      1
  deep learning             1
  experience                1
  experience natural        1


## 3. N-Gram Overlap for Resume Comparison

Jaccard overlap on **character** n-grams (via `analyzer="char"`) measures surface similarity between two strings: "data scientist python" vs "data analyst python" share most character bigrams even though they differ at the word level. This is a classic fuzzy-matching trick for near-duplicate text.

**What the code does:** `ngram_overlap()` builds character n-gram sets for both strings and returns `|A ∩ B| / |A ∪ B|`.
- Bigram overlap: `0.462`; trigram overlap: `0.385`.

**Try it:** the overlap is high but not 1 — the shared "data ... python" skeleton dominates, while the scientist/analyst difference shows up in the drop from 0.462 (bigrams) to 0.385 (trigrams). Lower n = more forgiving.

In [ ]:
def ngram_overlap(a, b, n=2):
    va = CountVectorizer(ngram_range=(n, n), analyzer="char").fit([a])
    vb = CountVectorizer(ngram_range=(n, n), analyzer="char").fit([b])
    a_ng = set(va.get_feature_names_out())
    b_ng = set(vb.get_feature_names_out())
    return len(a_ng & b_ng) / len(a_ng | b_ng) if (a_ng | b_ng) else 0

a = "data scientist python"
b = "data analyst python"
print(f"Bigram overlap: {ngram_overlap(a, b):.3f}")
print(f"Trigram overlap: {ngram_overlap(a, b, 3):.3f}")

## Summary: Use ngram_range=(1,2) for most resume tasks — captures bi-gram skills like 'machine learning'.

**The (1,2) range is the default: unigrams catch single skills, bigrams catch the multi-word ones that actually matter.**

Trigrams add context but multiply vocabulary and sparsity for little gain on resumes; character n-grams are reserved for fuzzy and typo-tolerant matching. Combined with TF-IDF weighting (Ch. 16), n-gram features are the classic classical-ML resume representation — and exactly the vectors Ch. 18 will measure with cosine similarity.